In [30]:
import os
import torch 
import pandas as pd
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import torch.optim as optim

In [31]:
class RunOrWalkNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(6, 4)  # Input layer: 6 neurons, Hidden Layer: 4 neurons
        self.fc2 = nn.Linear(4, 2)  # Hidden layer: 4 neurons, Output Layer: 2 neurons


    def forward(self, x):
        x = self.fc1(x)
        x = F.leaky_relu(x)
        x = self.fc2(x)
        output = F.softmax(x, dim=1)  # dimension 1: softmax function applied along second dimension of the tensor
        return output

In [32]:
class RunOrWalkDataset(Dataset):
    def __init__(self, file_path):
        # preprocessing done here
        self.data = pd.read_csv(file_path)
        self.data = self.data.drop(columns=['date', 'time', 'username', 'wrist'])
        self.data = self.data / np.linalg.norm(self.data, axis=0)
        self.labels = self.data.pop('activity').values
        scaler = StandardScaler()
        scaled_data = scaler.fit_transform(self.data)
        min_max_scaler = MinMaxScaler(feature_range=(0, 1))
        scaled_min_max_data = min_max_scaler.fit_transform(scaled_data)
        self.data = pd.DataFrame(scaled_min_max_data, columns=self.data.columns)


    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        features = torch.tensor(self.data.iloc[idx].values, dtype=torch.float)
        label = torch.tensor(self.labels[idx], dtype=torch.long)  # Assuming labels are integers
        return features, label


In [33]:
dataset = RunOrWalkDataset("./dataset.csv")
train_data, test_data = train_test_split(dataset, test_size=0.25, random_state=1334)  # my favourite number

In [34]:
batch_size = 32
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)


In [35]:
model = RunOrWalkNN()
optimizer = optim.SGD(model.parameters(), lr=0.2)
criterion = nn.CrossEntropyLoss()

In [36]:
epochs = 10

for epoch in range(epochs):
    model.train()  # Sets the model to training mode
    for inputs, labels in train_loader:
        optimizer.zero_grad()  # Zero the gradients
        outputs = model(inputs)  # Forward pass
        loss = criterion(outputs, labels)  # Calculate the loss
        loss.backward()  # Backward pass
        optimizer.step()  # Update weights

    model.eval()  # Sets the model to evaluation mode
    with torch.no_grad():  # Disable gradient calculation during testing
        correct = 0
        total = 0
        for inputs, labels in test_loader:
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)  # we dont need all the maximum of a tensor, hence _
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = correct / total

    print(f'Epoch [{epoch + 1}/{epochs}], Test Accuracy: {accuracy:.2%}')


Epoch [1/10], Test Accuracy: 100.00%
Epoch [2/10], Test Accuracy: 100.00%
Epoch [3/10], Test Accuracy: 100.00%
Epoch [4/10], Test Accuracy: 100.00%
Epoch [5/10], Test Accuracy: 100.00%
Epoch [6/10], Test Accuracy: 100.00%
Epoch [7/10], Test Accuracy: 100.00%
Epoch [8/10], Test Accuracy: 100.00%
Epoch [9/10], Test Accuracy: 100.00%
Epoch [10/10], Test Accuracy: 100.00%
